In [114]:
# Import the necessary libraries

import base64
import requests as rq
import json

In [115]:
# Define a function in order to obtain our personalised token

def get_oauth_token():
    '''
    This function will return our personalised token
    '''
    api_key = 'yvnm2w1s15oa2xp8j5ju422whxginj2l'   # Your API key provided by Idealista
    secret = 'tG20tgahxMPe'   # Your secred code provided by Idealista

    message = api_key + ":" + secret   # Combine the API key and the secret to get our personalised message

    auth = "Basic " + base64.b64encode(message.encode("ascii")).decode("ascii")   # Encode the message

    headers_dic = {"Authorization" : auth,
                   "Content-Type" : "application/x-www-form-urlencoded;charset=UTF-8"}   # Define our headers

    params_dic = {"grant_type" : "client_credentials",   # Define the request params
                  "scope" : "read"}

    r = rq.post("https://api.idealista.com/oauth/token",   # Perform the request with the api url, headers and params
                      headers = headers_dic,
                      params = params_dic)

    token = json.loads(r.text)['access_token']   # Obtain the personalised token, as a json

    return token

In [116]:
# This are the params we will use to filter our search

base_url = 'https://api.idealista.com/3.5/'     # Base search url
country = 'es'     # Search country (es, it, pt)
language = 'es'     # Search language (es, it, pt, en, ca)
max_items = '50'     # Max items per call, the maximum set by Idealista is 50
operation = 'rent'     # Kind of operation (sale, rent)
property_type = 'homes'     # Type of property (homes, offices, premises, garages, bedrooms)
order = 'priceDown'     # Order of the listings, consult documentation for all the available orders
center = '41.3825,2.176944'     # Coordinates of the search center
distance = '33520'     # Max distance from the center
sort = 'desc'     # How to sort the found items
bankOffer = 'false'     # If the owner is a bank. it works just for sales in spanain
minprice = '3001'
maxprice = '5000000000'     # Max price of the listings Barcelona center to Sitges: 33,52 km
province = 'Barcelona'
municipality = 'Barcelona'

In [117]:
# Define a function to obtain our search url

def define_search_url():
    '''
    This function will combine our params with the url, in order to create our own search url
    '''
    url = (base_url +
           country +
           '/search?operation=' + operation +
           '&maxItems=' + max_items +
           '&order=' + order +
           '&center=' + center +
           '&distance=' + distance +
           '&propertyType=' + property_type +
           '&sort=' + sort +
           '&numPage=%s' +  # Add %s as a placeholder for pagination
           '&maxPrice=' + maxprice +
           '&minPrice=' + minprice +
           '&language=' + language +
           '&municipality=' + municipality +
           '&province=' + province)

    return url

In [118]:
url = define_search_url()

In [119]:
def search_api(url):
    '''
    This function will use the token and url created previously, and return our search results.
    '''
    token = get_oauth_token()   #  Get the personalised token

    headers = {'Content-Type': 'Content-Type: multipart/form-data;',   # Define the search headers
               'Authorization' : 'Bearer ' + token}

    content = rq.post(url, headers = headers)   # Return the content from the request

    result = json.loads(content.text)   # Transform the result as a json file

    return result

In [120]:
# Since we need to give pagination to our search and this is our first search, we will set the pagination as 1
pagination = 1
first_search_url = url %(pagination)

In [121]:
# Proceed to do the search with the paginated url
results = search_api(first_search_url)

In [122]:
# First of all, we can extract 50 results/page, but there are more pages, so we have to define how many pages there are.

total_pages = results['totalPages']
total_pages

30

In [123]:
# Import the necessary libraries

import pandas as pd

In [124]:
def results_to_df(results):
    '''
    This function will save the json results as a dataframe and return the resulting dataframe
    '''
    df = pd.DataFrame.from_dict(results['elementList'])

    return df

In [125]:
def concat_df(df, df_tot):
    '''
    This function will take the main dataframe (df_tot), and concat it with the given individual dataframe,
    returning the main dataframe
    '''
    df_tot = pd.concat([df_tot, df], ignore_index=True)  # Concatenate and ignore original indices
    return df_tot

In [126]:
# Proceed to save the obtained results as a dataframe
df = results_to_df(results)

In [127]:
# Since we still don't have a main dataframe where we can store all the data, we will create an empty dataframe
df_tot = pd.DataFrame()
df_tot = concat_df(df, df_tot)

In [128]:
for i in range(2, total_pages):  #for i in range(2, total_pages):
    paginated_url = url % (i)  # Formatear la URL con el valor de i
    results = search_api(paginated_url)  # Obtener los resultados de búsqueda
    df = results_to_df(results)  # Guardar los resultados como un DataFrame
    df_tot = concat_df(df, df_tot)  # Concatenar los resultados al DataFrame principal

In [129]:
# Once we have all our data, we just need to save it as a csv file, we have created the following function for that:

file_path = 'idealista.csv'

def df_to_csv(df):
    '''
    This function will take a given dataframe and save it as a csv file
    '''
    df = df.reset_index()   # Reset the index in order to organise the records
    df.to_csv(file_path, index=False, sep=';', encoding='utf-8')  # Save it into a csv

In [130]:
# Run the function and you'll obtain a csv file with all the extracted data
df_to_csv(df_tot)

In [131]:
df2 = pd.read_csv('idealista.csv', sep=';', encoding='utf-8')

In [132]:
df2.head()

,index,propertyCode,thumbnail,externalReference,numPhotos,price,priceInfo,propertyType,operation,size,...,hasStaging,topNewDevelopment,topPlus,exterior,district,neighborhood,hasLift,highlight,floor,newDevelopmentFinished
0,0,103767484,https://img4.idealista.com/blur/WEB_LISTING/0/...,544-3000,15,3800.0,"{'price': {'amount': 3800.0, 'currencySuffix':...",chalet,rent,320.0,...,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,104376525,https://img4.idealista.com/blur/WEB_LISTING/0/...,W-02V8E7,22,5000.0,"{'price': {'amount': 5000.0, 'currencySuffix':...",penthouse,rent,115.0,...,False,False,False,True,Eixample,Sant Antoni,True,NaN,NaN,NaN
2,2,104975104,https://img4.idealista.com/blur/WEB_LISTING/0/...,W-02VXAB,22,4000.0,"{'price': {'amount': 4000.0, 'currencySuffix':...",flat,rent,73.0,...,False,False,False,True,Ciutat Vella,La Barceloneta,True,{'groupDescription': 'Destacado'},NaN,NaN
3,3,105215797,https://img4.idealista.com/blur/WEB_LISTING/0/...,AA052,19,3750.0,"{'price': {'amount': 3750.0, 'currencySuffix':...",flat,rent,123.0,...,False,False,False,True,Ciutat Vella,Sant Pere - Santa Caterina i la Ribera,True,{'groupDescription': 'Destacado'},1,NaN
4,4,36412002,https://img4.idealista.com/blur/WEB_LISTING/0/...,2769,44,7500.0,"{'price': {'amount': 7500.0, 'currencySuffix':...",flat,rent,247.0,...,False,False,False,True,Sant Martí,Diagonal Mar i el Front Marítim del Poblenou,True,NaN,NaN,NaN


In [133]:

# Definir la ruta y nombre del archivo de texto
file_path = 'idealista.txt'

def df_to_text(df):
    '''
    Esta función toma un DataFrame dado y lo guarda como un archivo de texto (.txt)
    con cada fila del DataFrame siendo una línea en el archivo de texto,
    y separando los valores por punto y coma (;).
    '''
    with open(file_path, 'w', encoding='utf-8') as file:
        for index, row in df.iterrows():
            # Construir la línea con los valores de cada fila separados por punto y coma (;)
            line = ';'.join(str(value) for value in row) + '\n'
            file.write(line)

In [134]:
# Run the function and you'll obtain a csv file with all the extracted data
df_to_text(df_tot)